In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics

In [ ]:
!apt-get install -y fonts-nanum > /dev/null 2>&1

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from ultralytics import YOLO
import pandas as pd
import os

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
BASE_PATH = "/content/drive/MyDrive/medication-detection"

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data=f"{BASE_PATH}/dataset_combined/data.yaml",
    epochs=60,
    patience=20,
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.00176,
    weight_decay=0.0,
    momentum=0.9349,
    box=7.70337,
    cls=0.57286,
    dfl=2.02811,
    project=f"{BASE_PATH}/runs",
    name="exp_combined_v8n",
    exist_ok=True,
    mosaic=0.85211,
    mixup=0.00489,
    flipud=0.00328,
    fliplr=0.0,
    hsv_h=0.0171,
    hsv_s=0.60513,
    hsv_v=0.44902,
    translate=0.0,
    scale=0.00162,
    erasing=0.0,
    auto_augment="none",
)

In [ ]:
results = pd.read_csv(f"{BASE_PATH}/runs/exp_combined_v8n/results.csv")
results.columns = results.columns.str.strip()

print("AI Hub 병합 데이터 최고 성능")
print(f"mAP@50:    {results['metrics/mAP50(B)'].max():.4f}")
print(f"mAP@50-95: {results['metrics/mAP50-95(B)'].max():.4f}")
print(f"Precision: {results['metrics/precision(B)'].max():.4f}")
print(f"Recall:    {results['metrics/recall(B)'].max():.4f}")

In [ ]:
import shutil
shutil.copytree(f"{BASE_PATH}/test_images", "/content/test_images", dirs_exist_ok=True)

In [ ]:
import gc
import cv2
import torch

model = YOLO(f"{BASE_PATH}/runs/exp_combined_v8n/weights/best.pt")

test_img_dir = "/content/test_images"
test_files = sorted(os.listdir(test_img_dir))

sorted_cat_ids = sorted([1900, 2483, 3351, 3483, 3544, 4543, 12081, 12247, 12778, 13395,
                          13900, 16232, 16262, 16548, 16551, 16688, 18147, 18357, 19232,
                          19552, 19607, 19861, 20014, 20238, 20877, 21325, 21771, 22074,
                          22347, 22362, 24850, 25367, 25438, 25469, 27733, 27777, 27926,
                          27993, 28763, 29345, 29451, 29667, 30308, 31863, 31885, 32310,
                          33009, 33208, 33880, 34597, 35206, 36637, 38162, 41768, 3832, 3743])
idx_to_cat_id = {idx: cat_id for idx, cat_id in enumerate(sorted_cat_ids)}

pad_size = 1280
out_size = 640
save_path = f"{BASE_PATH}/submission.csv"

rows = []
annotation_id = 1

for i, fname in enumerate(test_files):
    if not fname.endswith('.png'):
        continue

    img_path = f"{test_img_dir}/{fname}"
    image_id = int(fname.replace('.png', ''))

    orig = cv2.imread(img_path)
    oh, ow = orig.shape[:2]

    pad_w = pad_size - ow
    pad_h = pad_size - oh
    left = pad_w // 2
    top = pad_h // 2

    padded = cv2.copyMakeBorder(orig, top, pad_h - top, left, pad_w - left,
                                 cv2.BORDER_CONSTANT, value=(114, 114, 114))
    resized = cv2.resize(padded, (out_size, out_size))

    results = model.predict(resized, imgsz=out_size, conf=0.25, verbose=False)[0]

    for box in results.boxes:
        cls_idx = int(box.cls[0])

        # 118개 클래스 중 캐글 56개(idx 0~55)만 사용
        if cls_idx >= 56:
            continue

        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])

        scale = pad_size / out_size
        x1 = x1 * scale - left
        y1 = y1 * scale - top
        x2 = x2 * scale - left
        y2 = y2 * scale - top

        x1 = max(0, min(x1, ow))
        y1 = max(0, min(y1, oh))
        x2 = max(0, min(x2, ow))
        y2 = max(0, min(y2, oh))

        bw = x2 - x1
        bh = y2 - y1

        if bw <= 0 or bh <= 0:
            continue

        category_id = idx_to_cat_id[cls_idx]

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "bbox_x": round(x1),
            "bbox_y": round(y1),
            "bbox_w": round(bw),
            "bbox_h": round(bh),
            "score": round(conf, 4)
        })
        annotation_id += 1

    del orig, padded, resized, results
    gc.collect()
    torch.cuda.empty_cache()

submission_df = pd.DataFrame(rows)
submission_df.to_csv(save_path, index=False)
print(f"완료! 총 예측 수: {len(submission_df)}")